In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from timeit import default_timer as timer

import os

# Point XLA to your CUDA/libdevice directory
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'

import tensorflow as tf
tf.config.optimizer.set_jit(False)

dataset = "/media/yassin/Nuevo vol/Datasets/spanish_traffic/Classification/samples"

labelfile = pd.read_csv("/media/yassin/Nuevo vol/Datasets/spanish_traffic/Classification/" + "gt_spanish_dataset.csv")

2025-09-14 21:19:01.985097: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
labelfile.head()

,image,width,height,class_id,class_name
0,Image00001.jpg,174,174,1,R-1
1,Image00002.jpg,240,240,1,R-1
2,Image00003.jpg,320,320,1,R-1
3,Image00004.jpg,120,120,1,R-1
4,Image00005.jpg,86,86,1,R-1


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models  # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint # type: ignore

img_size = (128, 128)
batch_size = 32
validation_split = 0.1
seed = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset,
    validation_split=validation_split,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

# Load validation / test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset,
    validation_split=validation_split,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

Found 1478 files belonging to 99 classes.
Using 1331 files for training.


I0000 00:00:1757877543.430996  114045 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5160 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


Found 1478 files belonging to 99 classes.
Using 147 files for validation.


In [4]:
num_classes = 99 

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = models.Sequential([
    layers.InputLayer(input_shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    data_augmentation,        
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint('model.keras', monitor='val_accuracy', save_best_only=True)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=100,
    callbacks=[checkpoint]
)

best_model = tf.keras.models.load_model('model.keras')
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

/home/yassin/miniconda3/envs/tf/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/100


2025-09-14 21:19:06.980785: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.1886 - loss: 3.7755 - val_accuracy: 0.3810 - val_loss: 3.0860
Epoch 2/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.4020 - loss: 2.7428 - val_accuracy: 0.4354 - val_loss: 2.4634
Epoch 3/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.4718 - loss: 2.3165 - val_accuracy: 0.4966 - val_loss: 2.3533
Epoch 4/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.5192 - loss: 1.9778 - val_accuracy: 0.5646 - val_loss: 1.9639
Epoch 5/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5785 - loss: 1.7083 - val_accuracy: 0.5510 - val_loss: 1.9420
Epoch 6/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.6176 - loss: 1.5330 - val_accuracy: 0.5782 - val_loss: 1.7299
Epoch 7/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.6521 - loss: 1.3748 - val_accuracy: 0.6599 - val_loss: 1.6087
Epoch 8/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.6672 - loss: 1.2702 - val_accuracy: 0.6667 - val_